# Load raw datasets

In [118]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [222]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.eda_utils import (
    clean_headers,
    load_gh_aw_config_files,
    load_run_summaries,
    load_workflows_info,
    create_gh_aw_id,
    parse_colected_date,
    parse_md_SHA,
    parse_md_url,
    parse_repo,
    parse_workflow,
    parse_yml_SHA,
    parse_yml_filename,
    parse_yml_url,
)

In [142]:
df = load_gh_aw_config_files("data/gh-aw_raw")
df.shape

(5501, 2)

In [143]:
df.head()

,workflow_markdown,workflow_yml
0,# Repo: 0GiS0/github-agentic-workflows-demos\n...,# Repo: 0GiS0/github-agentic-workflows-demos\n...
1,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...
2,# Repo: AIS-Commercial-Business-Unit/RiskInsur...,# Repo: AIS-Commercial-Business-Unit/RiskInsur...
3,# Repo: AdaCore/z3\n# Workflow: daily-backlog-...,# Repo: AdaCore/z3\n# Workflow: daily-backlog-...
4,# Repo: AdaCore/z3\n# Workflow: pr-fix\n# MD U...,# Repo: AdaCore/z3\n# Workflow: pr-fix\n# MD U...


# Parse fields and remove metadata in the header

In [ ]:
df["repo"] = df["workflow_markdown"].apply(parse_repo)
df["md_file_name"] = df["workflow_markdown"].apply(parse_workflow) # workflow filed in the metada is just the .md file name. Use this name for another field to map with the actual workflow name
df["md_url"] = df["workflow_markdown"].apply(parse_md_url)
df["yml_url"] = df["workflow_yml"].apply(parse_yml_url)
df["md_sha"] = df["workflow_markdown"].apply(parse_md_SHA)
df["yml_sha"] = df["workflow_yml"].apply(parse_yml_SHA)
df["collected_date"] = df["workflow_markdown"].apply(parse_colected_date)
df["workflow_markdown_without_header"] = df["workflow_markdown"].apply(clean_headers)
df["workflow_yml_without_header"] = df["workflow_yml"].apply(clean_headers)
df["yml_filename"] = df["yml_url"].apply(parse_yml_filename) # use this field to map with workflow runs 
df["gh_aw_id"] = df["workflow_markdown_without_header"].apply(create_gh_aw_id)
df['id'] = df['repo'] + df['gh_aw_id'] # this id should be unique and use to join
# we treat each gh_wh in each repo is a seperated analytic unit even if they have the same .md file. Because we will need to analyze the workflows which are different by repos 

Note: I found 1 .md content file can have multiple .yml content files. Quick check reason is the update of the gh_aw compiler version. So it can be a source of runtime error. But for first analysis iteration. keep it simpple that we create id based on the .md content only

In [145]:
df.head(2)

,workflow_markdown,workflow_yml,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,collected_date,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id
0,# Repo: 0GiS0/github-agentic-workflows-demos\n...,# Repo: 0GiS0/github-agentic-workflows-demos\n...,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,2026-03-15T18:34:23.258706,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...
1,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,2026-03-15T18:34:24.714772,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...


In [146]:
df["gh_aw_id"].nunique()

3203

In [148]:
df['id'].nunique() # which mean same .md file can sit in different repos

3263

In [149]:
df['collected_date'] = df['collected_date'].astype('datetime64[ns]')

In [150]:
df['ym'] = df['collected_date'].dt.strftime('%Y%m')

In [151]:
df.isnull().sum()

workflow_markdown                   0
workflow_yml                        0
repo                                0
md_file_name                        0
md_url                              0
yml_url                             0
md_sha                              0
yml_sha                             0
collected_date                      0
workflow_markdown_without_header    0
workflow_yml_without_header         0
yml_filename                        0
gh_aw_id                            0
id                                  0
ym                                  0
dtype: int64

In [152]:
df.head(2)

,workflow_markdown,workflow_yml,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,collected_date,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id,ym
0,# Repo: 0GiS0/github-agentic-workflows-demos\n...,# Repo: 0GiS0/github-agentic-workflows-demos\n...,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,2026-03-15 18:34:23.258706,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...,202603
1,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,2026-03-15 18:34:24.714772,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...,202603


In [153]:
df.shape

(5501, 15)

In [155]:
df['md_url'].nunique() # we cannot use this as a key because its linked to the commit SHA at the collect time

4077

In [156]:
df['ym'].value_counts()

ym
202606    2263
202605     904
202609     827
202608     769
202604     425
202603     313
Name: count, dtype: int64

In [52]:
df.groupby(['md_url'])['ym'].count()

md_url
https://github.com/0GiS0/0gis0/blob/0d9f36cf611a64484a4ede5e8943bdd82dcc7d9e/.github/workflows/smart-labeler.md                                        1
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/continuous-documentation.md    4
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/issue-enhancer.md              1
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/label-beautifier.md            4
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/the-suggester.md               2
                                                                                                                                                      ..
https://github.com/zircote/rlm-rs/blob/72eaedc6304e6d239748cf9e976148227774

because many workflows are duplicated, we use dropduplicate and keep the first collected date only

In [70]:
df.columns

Index(['workflow_markdown', 'workflow_yml', 'repo', 'workflow', 'md_url',
       'yml_url', 'md_SHA', 'yml_SHA', 'collected_date',
       'workflow_markdown_without_header', 'workflow_yml_without_header',
       'ym'],
      dtype='str')

In [158]:
non_dup_df = df.drop(['workflow_markdown', 'workflow_yml', 'collected_date'], axis=1) # we drop this as the header of the files contains collected date which makes them different
non_dup_df = non_dup_df.sort_values(by='ym', ascending=True) # then we keep first in the drop
non_dup_df = non_dup_df.drop_duplicates(subset=['id'], keep='first')

In [159]:
non_dup_df.shape

(3263, 12)

In [160]:
non_dup_df.head(2)

,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id,ym
0,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...,202603
1,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...,202603


In [161]:
assert non_dup_df.groupby(['id'])['ym'].count().max() == 1 # assert to make sure the df is unique by md_url

In [162]:
non_dup_df.head(2)

,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id,ym
0,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...,202603
1,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...,202603


In [163]:
non_dup_df.ym.value_counts()

ym
202606    1204
202605     644
202608     428
202609     352
202604     322
202603     313
Name: count, dtype: int64

In [168]:
with pd.option_context('display.max_colwidth', None):
    display(non_dup_df[non_dup_df['md_file_name'] == 'issue-quality-enhancer']['md_url'])

1    https://github.com/0GiS0/jsconf-2026/blob/180aa116c1a817d04f356e592a4f2a028d6d2ea0/.github/workflows/issue-quality-enhancer.md
Name: md_url, dtype: str

In [174]:
assert non_dup_df.shape[0]==(non_dup_df['id'].nunique())

In [176]:
non_dup_df.shape

(3263, 12)

In [183]:
non_dup_df[['repo']].drop_duplicates().reset_index(drop=True).to_csv('unique_repo.csv', index=False)

# Workflows information 

we have 980 repos, these are all the workflows in the repo, we're mapping to get only the workflows that created by GH-AW

In [185]:
workflows_df = load_workflows_info("data/workflow_info/workflows.csv")
workflows_df.shape

(14793, 13)

In [187]:
workflows_df.head(2)

,repo,yml_filename,state,path,name,workflow_id,created_at,updated_at,html_url,badge_url,url,node_id,deleted_at
0,0GiS0/0gis0,continuous-docs.lock.yml,active,.github/workflows/continuous-docs.lock.yml,Continuous Documentation,331774783,2026-08-11T05:34:01.000-04:00,2026-08-11T07:52:56.000-04:00,https://github.com/0GiS0/0gis0/blob/main/.gith...,https://github.com/0GiS0/0gis0/workflows/Conti...,https://api.github.com/repos/0GiS0/0gis0/actio...,W_kwDOGVl6BM4Txns_,NaN
1,0GiS0/0gis0,generate-snake.yml,active,.github/workflows/generate-snake.yml,🐍 Generate Snake Animation,214319767,2025-12-09T06:35:26.000-05:00,2026-01-25T08:04:50.000-05:00,https://github.com/0GiS0/0gis0/blob/main/.gith...,https://github.com/0GiS0/0gis0/workflows/%F0%9...,https://api.github.com/repos/0GiS0/0gis0/actio...,W_kwDOGVl6BM4MxkKX,NaN


In [ ]:
workflows_df.state.value_counts()

state
active                 13102
disabled_manually       1149
disabled_inactivity      414
disabled_fork            127
disabled_admin             1
Name: count, dtype: int64

- disabled_manually: people turned it off
- disabled_inactviity: github turned it off because of inactivity
- disabled_fork: the workflow was originally created in other repo --> fork auto disable the workflow
- disabled_admin: github org admin turned it off


In [209]:
13102/workflows_df.shape[0] # this is all workflows status including gh-aw

0.8856891773135943

In [204]:
non_gh_aw_workflows_df = workflows_df.merge(non_dup_df, on=['repo', 'yml_filename'], how='left_anti')

In [206]:
non_gh_aw_workflows_df.state.value_counts()

state
active                 11533
disabled_manually        794
disabled_inactivity      275
disabled_fork             87
disabled_admin             1
Name: count, dtype: int64

In [ ]:
11533/non_gh_aw_workflows_df.shape[0] # exclude gh-aw we see % active workflows is high

0.9088258471237195

In [211]:
non_gh_aw_workflows_df.shape

(12690, 23)

In [213]:
non_gh_aw_workflows_df.sample(373)[['repo', 'yml_filename']].to_csv("non_gh_aw_workflows_df.csv", index=False)

In [197]:
gh_aw_workflows_df = non_dup_df.merge(workflows_df, on=['yml_filename', 'repo'])

In [198]:
gh_aw_workflows_df.head()

,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,...,path,name,workflow_id,created_at,updated_at,html_url,badge_url,url,node_id,deleted_at
0,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,...,.github/workflows/label-beautifier.lock.yml,💅🏻 Label beautifier,237822369,2026-02-23T18:21:16.000-05:00,2026-02-23T18:21:16.000-05:00,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,https://api.github.com/repos/0GiS0/github-agen...,W_kwDORXMKWM4OLOGh,NaN
1,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,...,.github/workflows/issue-quality-enhancer.lock.yml,Mejorador de Issues,244335156,2026-03-10T17:10:12.000-04:00,2026-03-10T17:10:12.000-04:00,https://github.com/0GiS0/jsconf-2026/blob/main...,https://github.com/0GiS0/jsconf-2026/workflows...,https://api.github.com/repos/0GiS0/jsconf-2026...,W_kwDORg2AP84OkEI0,NaN
2,AdaCore/z3,daily-backlog-burner,https://github.com/AdaCore/z3/blob/24563ee6bd3...,https://github.com/AdaCore/z3/blob/24563ee6bd3...,eca1fc3417154bea2edc0e05c9e2061d9e02d23f,5dfd11104c31c8b2d9d035b0211152b11ff921e0,---\non:\n workflow_dispatch:\n schedule...,# This file was automatically generated by gh-...,daily-backlog-burner.lock.yml,65414a3ad84034ca5c1f91b08ea624360e6d9c26d7794a...,...,.github/workflows/daily-backlog-burner.lock.yml,Daily Backlog Burner,221693066,2026-01-08T03:57:57.000-05:00,2026-01-08T03:57:57.000-05:00,https://github.com/AdaCore/z3/blob/master/.git...,https://github.com/AdaCore/z3/workflows/Daily%...,https://api.github.com/repos/AdaCore/z3/action...,W_kwDOA_xSGs4NNsSK,NaN
3,AdaCore/z3,pr-fix,https://github.com/AdaCore/z3/blob/24563ee6bd3...,https://github.com/AdaCore/z3/blob/24563ee6bd3...,146c9eb1f72244cae8c06be13393d70020798045,2e2679e64ed5ee71c84fcda5c116a71854d56c82,---\non:\n command:\n name: pr-fix\n reac...,# This file was automatically generated by gh-...,pr-fix.lock.yml,95b38dc567656ed8b3b7df483525824b27b0c1876cab1f...,...,.github/workflows/pr-fix.lock.yml,PR Fix,221693072,2026-01-08T03:57:57.000-05:00,2026-01-08T03:57:57.000-05:00,https://github.com/AdaCore/z3/blob/master/.git...,https://github.com/AdaCore/z3/workflows/PR%20F...,https://api.github.com/repos/AdaCore/z3/action...,W_kwDOA_xSGs4NNsSQ,NaN
4,Azure-Samples/Legacy-Modernization-Agents,documentation-audit,https://github.com/Azure-Samples/Legacy-Modern...,https://github.com/Azure-Samples/Legacy-Modern...,60440c1807638585032c2a519cff8c400d739103,4f0de00ac3b5fc07430f47b982acac4b92d934d2,---\ndescription: Weekly full audit comparing ...,#\n# ___ _ _ \n# ...,documentation-audit.lock.yml,19e2d4b4a0dd2e6bd5eb30df786e90b5f9096cdb450623...,...,.github/workflows/documentation-audit.lock.yml,Documentation Full Audit,236598858,2026-02-20T07:45:56.000-05:00,2026-02-20T07:45:56.000-05:00,https://github.com/Azure-Samples/Legacy-Modern...,https://github.com/Azure-Samples/Legacy-Modern...,https://api.github.com/repos/Azure-Samples/Leg...,W_kwDOO2KiOc4OGjZK,NaN


In [199]:
gh_aw_workflows_df.state.value_counts()

state
active                 2254
disabled_manually       427
disabled_inactivity     142
disabled_fork            40
Name: count, dtype: int64

In [201]:
2254/gh_aw_workflows_df.shape[0]

0.7872860635696821

In [202]:
gh_aw_workflows_df.shape, gh_aw_workflows_df['id'].nunique()

((2863, 23), 2863)

In [220]:
gh_aw_workflows_df[['repo', 'yml_filename']].drop_duplicates().sample(325).to_csv("gh_aw_workflows_df.csv", index=False)

In [217]:
gh_aw_workflows_df.groupby(['repo', 'yml_filename'])['md_file_name'].count().sort_values()

repo                                     yml_filename                           
0GiS0/0gis0                              smart-labeler.lock.yml                     1
kaushal-waygood/agentic-self-healing-ci  daily-safe-outputs-conformance.lock.yml    1
                                         daily-safe-output-optimizer.lock.yml       1
                                         daily-issues-report.lock.yml               1
                                         daily-doc-updater.lock.yml                 1
                                                                                   ..
github/gh-aw                             audit-workflows.lock.yml                   5
                                         daily-architecture-diagram.lock.yml        5
                                         archie.lock.yml                            5
                                         issue-monster.lock.yml                     5
                                         daily-community-at

what happened to the 21% inactive? why they have double inactivity rate compared to non-gh-aw (only 10% inactivity)?

# Check error rate of all runs of workflows

Compare the error rate of all runs of gh-aw vs non-gh-aw. Because its large, sampling already applied to have statistic significant

In [223]:
gh_aw_runs_df = load_run_summaries("gh-aw")
non_gh_aw_runs_df = load_run_summaries("non-gh-aw")
gh_aw_runs_df.shape, non_gh_aw_runs_df.shape

((967, 19), (643, 19))

In [225]:
gh_aw_runs_df.head()

,owner,repo,run_id,name,display_title,status,conclusion,event,head_branch,head_sha,path,html_url,created_at,updated_at,run_attempt,job_count,logs_skipped,extracted_at,output_dir
0,dsyme,raft-lean-squad,23264228328,Lean Squad,Lean Squad,completed,failure,workflow_dispatch,main,1ba922ef24c54c0854d8fa3ab777ec3575840b76,.github/workflows/lean-squad.lock.yml,https://github.com/dsyme/raft-lean-squad/actio...,2026-03-18T19:56:33Z,2026-03-18T19:58:55Z,1,5,False,2026-10-05T14:24:07Z,data/workflow_details/gh_aw_workflows/dsyme__r...
1,dsyme,raft-lean-squad,23264324462,Lean Squad,Lean Squad,completed,failure,workflow_dispatch,main,1ba922ef24c54c0854d8fa3ab777ec3575840b76,.github/workflows/lean-squad.lock.yml,https://github.com/dsyme/raft-lean-squad/actio...,2026-03-18T19:58:58Z,2026-03-18T20:01:30Z,1,5,False,2026-10-05T14:24:02Z,data/workflow_details/gh_aw_workflows/dsyme__r...
2,dsyme,raft-lean-squad,23264431912,Lean Squad,Lean Squad,completed,failure,workflow_dispatch,main,1ba922ef24c54c0854d8fa3ab777ec3575840b76,.github/workflows/lean-squad.lock.yml,https://github.com/dsyme/raft-lean-squad/actio...,2026-03-18T20:01:41Z,2026-03-18T20:04:02Z,1,5,False,2026-10-05T14:24:01Z,data/workflow_details/gh_aw_workflows/dsyme__r...
3,dsyme,raft-lean-squad,23264510315,Lean Squad,Lean Squad,completed,success,workflow_dispatch,main,1ba922ef24c54c0854d8fa3ab777ec3575840b76,.github/workflows/lean-squad.lock.yml,https://github.com/dsyme/raft-lean-squad/actio...,2026-03-18T20:03:39Z,2026-03-18T20:11:21Z,1,5,False,2026-10-05T14:24:02Z,data/workflow_details/gh_aw_workflows/dsyme__r...
4,dsyme,raft-lean-squad,23264814550,Lean Squad,Lean Squad,completed,cancelled,workflow_dispatch,main,1ba922ef24c54c0854d8fa3ab777ec3575840b76,.github/workflows/lean-squad.lock.yml,https://github.com/dsyme/raft-lean-squad/actio...,2026-03-18T20:11:23Z,2026-03-18T20:20:50Z,1,5,False,2026-10-05T14:24:03Z,data/workflow_details/gh_aw_workflows/dsyme__r...


In [226]:
gh_aw_runs_df['conclusion'].value_counts()

conclusion
success      556
skipped      240
failure      158
cancelled     13
Name: count, dtype: int64

In [227]:
556/gh_aw_runs_df.shape[0]

0.5749741468459152

In [228]:
non_gh_aw_runs_df.conclusion.value_counts()

conclusion
success            531
failure             91
action_required      9
cancelled            8
startup_failure      4
Name: count, dtype: int64

In [230]:
531/non_gh_aw_runs_df.shape[0]

0.8258164852255054